# Notebook 03 — Network Topology and Properties

**Module:** 12 — Systems and Network Biology  
**Tier:** 2 — Working competence  
**Notebook:** 03 of 12  
**Time estimate:** 75 minutes

> Knowing the degree sequence isn't enough. The clustering coefficient and
> average path length reveal whether your network is a lattice, a random graph,
> or a small world — and only one of these describes most real biological networks.

---
## Step 1 — Motivation

In 1998, Watts and Strogatz showed that social networks, neural networks, and power
grids all share two surprising properties: very short average path lengths (like random
graphs) AND very high clustering (like lattices). This is the **small-world** property,
and it has deep implications for how signals propagate in biological networks.

---
## Step 2 — Intuition

**Clustering coefficient of a node $v$:** What fraction of $v$'s neighbors are also
neighbors of each other? A high clustering coefficient means "my friends are friends
with each other" — dense local triangles.

**Average path length:** The average number of hops between any two nodes. Random
graphs have $\langle d \rangle \sim \log N / \log \langle k \rangle$.

**Small-world:** High clustering (close to lattice) + short paths (close to random).
Most real biological networks are small-world: this means perturbations propagate
rapidly but local circuits are dense and redundant.

**Erdős-Rényi (ER) model:** Connect each pair with probability $p$. The null model
for biological networks — real networks are almost always different in specific ways.

---
## Step 3 — Biological Background

**Clustering in biology:** Proteins that work in the same complex tend to interact
with the same set of partners (high clustering). Metabolic enzymes in the same
pathway share many substrates (high clustering in metabolite-metabolite networks).

**Short paths in biology:** The average path length in the human PPI network is ~4–5
hops for ~5000 proteins. A signal or perturbation can propagate from any protein to
any other in surprisingly few steps — this is why cancer driver mutations in hub
proteins can have system-wide effects.

**Network diameter:** The longest shortest path. The diameter of the *C. elegans*
connectome (~300 neurons, ~7000 connections) is only 5.

**Transitivity (global clustering coefficient):**
$$C_{\text{global}} = \frac{3 \times \text{number of triangles}}{\text{number of connected triples}}$$

This is the fraction of "open triplets" that are closed into triangles — a single
number summarizing how cliquey the whole network is.

---
## Step 4 — Mathematical Explanation

**Local clustering coefficient:**
$$C_v = \frac{|\{(u,w) : u,w \in N(v), (u,w) \in E\}|}{k_v (k_v - 1) / 2}$$

where $N(v)$ is the neighbor set of $v$ and $k_v = |N(v)|$.
Numerator: edges actually present among neighbors.  
Denominator: maximum possible edges among $k_v$ neighbors = $\binom{k_v}{2}$.

**Average local clustering coefficient:**
$$\langle C \rangle = \frac{1}{n} \sum_v C_v$$

**Average path length:**
$$\langle d \rangle = \frac{1}{n(n-1)} \sum_{u \neq v} d(u,v)$$

Computed via BFS from every node: $O(n(n + m))$.

**Small-world index (normalized):**
$$\sigma = \frac{C / C_{\text{rand}}}{\langle d \rangle / \langle d \rangle_{\text{rand}}}$$

where $C_{\text{rand}} \approx \langle k \rangle / n$ and $\langle d \rangle_{\text{rand}} \approx \ln n / \ln \langle k \rangle$.
$\sigma > 1$ indicates small-world structure.

In [ ]:
# Step 6 — Python: Topology metrics from scratch
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

# ---- Utility: generate Erdős-Rényi random graph ----
def erdos_renyi(n, p, seed=42):
    rng = np.random.default_rng(seed)
    edges = []
    adj = {i: set() for i in range(n)}
    for i in range(n):
        for j in range(i+1, n):
            if rng.random() < p:
                edges.append((i, j))
                adj[i].add(j)
                adj[j].add(i)
    return adj, edges

# ---- Local clustering coefficient ----
def clustering_coefficient(adj, v):
    neighbors = list(adj[v])
    k = len(neighbors)
    if k < 2:
        return 0.0
    triangles = 0
    for i in range(k):
        for j in range(i+1, k):
            if neighbors[j] in adj[neighbors[i]]:
                triangles += 1
    return triangles / (k * (k-1) / 2)

def average_clustering(adj):
    nodes = list(adj.keys())
    return np.mean([clustering_coefficient(adj, v) for v in nodes])

# ---- BFS shortest path from source ----
def bfs_distances(adj, source):
    dist = {source: 0}
    queue = deque([source])
    while queue:
        u = queue.popleft()
        for v in adj[u]:
            if v not in dist:
                dist[v] = dist[u] + 1
                queue.append(v)
    return dist

def average_path_length(adj):
    nodes = list(adj.keys())
    total, count = 0, 0
    for u in nodes:
        dists = bfs_distances(adj, u)
        for v, d in dists.items():
            if v != u:
                total += d
                count += 1
    return total / count if count > 0 else 0

# ---- Compare three network types ----
n = 50
p = 0.1  # ER probability → mean degree ~4.9

# 1. ER random graph
adj_er, _ = erdos_renyi(n, p, seed=1)

# 2. Ring lattice (k=4)
def ring_lattice(n, k=4):
    adj = {i: set() for i in range(n)}
    for i in range(n):
        for j in range(1, k//2 + 1):
            adj[i].add((i+j) % n)
            adj[(i+j) % n].add(i)
    return adj

adj_lat = ring_lattice(n, k=4)

# 3. Watts-Strogatz small-world (rewire 5% of lattice edges)
def watts_strogatz(n, k=4, p_rewire=0.05, seed=42):
    rng = np.random.default_rng(seed)
    adj = ring_lattice(n, k)
    # Rewire each edge with probability p_rewire
    for i in range(n):
        for j in range(1, k//2 + 1):
            target = (i + j) % n
            if rng.random() < p_rewire:
                # Remove old edge
                adj[i].discard(target)
                adj[target].discard(i)
                # Add new random edge
                candidates = [v for v in range(n) if v != i and v not in adj[i]]
                if candidates:
                    new_v = rng.choice(candidates)
                    adj[i].add(new_v)
                    adj[new_v].add(i)
    return adj

adj_sw = watts_strogatz(n, k=4, p_rewire=0.1)

networks = {'ER random': adj_er, 'Ring lattice': adj_lat, 'Small-world (WS)': adj_sw}
print(f'{'Network':<25} {'<k>':<8} {'<C>':<10} {'<d>':<8}')
print('-' * 55)
for name, adj in networks.items():
    deg = np.mean([len(adj[v]) for v in adj])
    C = average_clustering(adj)
    d = average_path_length(adj)
    print(f'{name:<25} {deg:<8.2f} {C:<10.4f} {d:<8.4f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Simple spring-layout helper
def simple_layout(adj, seed=42):
    n = len(adj)
    rng = np.random.default_rng(seed)
    theta = np.linspace(0, 2*np.pi, n, endpoint=False)
    pos = np.column_stack([np.cos(theta), np.sin(theta)])
    return pos

labels = ['A. ER Random', 'B. Ring Lattice', 'C. Small-World (WS)']
for ax, (name, adj), label in zip(axes, networks.items(), labels):
    nodes = sorted(adj.keys())
    pos = simple_layout(adj)
    # Draw edges
    for u in adj:
        for v in adj[u]:
            if u < v:
                ax.plot([pos[u,0], pos[v,0]], [pos[u,1], pos[v,1]],
                          'grey', lw=0.3, alpha=0.4)
    # Draw nodes
    C_vals = [clustering_coefficient(adj, v) for v in nodes]
    sc = ax.scatter(pos[:,0], pos[:,1], c=C_vals, cmap='plasma',
                      s=80, zorder=2, vmin=0, vmax=1)
    deg = np.mean([len(adj[v]) for v in adj])
    C_mean = np.mean(C_vals)
    d_mean = average_path_length(adj)
    ax.set_title(f'{label}\n<k>={deg:.1f}, <C>={C_mean:.3f}, <d>={d_mean:.2f}', fontsize=8)
    ax.axis('off')
    plt.colorbar(sc, ax=ax, label='C_v', shrink=0.7)

plt.suptitle('Module 12 NB03: Network Topology\n(color = local clustering coefficient)',
               fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('network_topology.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Compute the **global** clustering coefficient (transitivity) for each of the three
   networks. How does it differ from the average local clustering coefficient?
2. Implement the small-world index $\sigma$. Which network has $\sigma > 1$?
3. What happens to average path length as you increase the rewiring probability in the
   Watts-Strogatz model from 0 to 1? (*The Watts-Strogatz transition.*)
4. In a PPI network with 5000 proteins and mean degree 10, estimate the average path
   length assuming ER-like structure.

---
## Step 10 — Quiz

1. What does the local clustering coefficient of a node measure?
2. What are the two defining properties of a small-world network?
3. Why is the ER model used as a null model for biological networks?
4. What does a short average path length imply for information propagation in a network?
5. How is the Watts-Strogatz model constructed?

---
## Step 12 — Reflection

> *[In one paragraph: explain why the small-world property matters specifically for
> a protein–protein interaction network. What biological advantage does it confer?]*

---
*Next: `04_scale_free_networks_and_barabasi_albert.ipynb`*